In [1]:
import numpy as np
from numpy import linalg as LA

from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input


In [2]:
class VGGNet:
    def __init__(self):
        # weights: 'imagenet'
        # pooling: 'max' or 'avg'
        # input_shape: (width, height, 3), width and height should >= 48
        self.input_shape = (224, 224, 3)
        self.weight = 'imagenet'
        self.pooling = 'max'
        self.model = VGG16(weights = self.weight, input_shape = (self.input_shape[0], self.input_shape[1], self.input_shape[2]), pooling = self.pooling, include_top = False)
        self.model.predict(np.zeros((1, 224, 224 , 3)))


    '''
    Use vgg16 model to extract features
    Output normalized feature vector
    '''
    def extract_feat(self, img_path):
        img = image.load_img(img_path, target_size=(self.input_shape[0], self.input_shape[1]))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        feat = self.model.predict(img)
        norm_feat = feat[0]/LA.norm(feat[0])
        return norm_feat

In [4]:
import os
import h5py
import numpy as np

In [6]:
images_path ="all_images/"
img_list = [os.path.join(images_path,f) for f in os.listdir(images_path)]
img_list

['all_images/4.png',
 'all_images/horse2.jpg',
 'all_images/zebra2.jpg',
 'all_images/chihuahua.JPG',
 'all_images/irish_terrier.JPG',
 'all_images/1.png',
 'all_images/horse1.jpg',
 'all_images/2.png',
 'all_images/tiger1.jpg',
 'all_images/monkey1.jpg',
 'all_images/monkey2.jpg',
 'all_images/zebra1.jpg',
 'all_images/tiger2.jpg']

In [7]:
model = VGGNet()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 860ms/step


In [8]:
path = "all_images/"

feats = []
names = []

for im in os.listdir(path):
    print("Extracting features from image - ", im)
    X = model.extract_feat(path+im)

    feats.append(X)
    names.append(im)

feats = np.array(feats)

output = "VGG16Features.h5"

print(" writing feature extraction results to h5 file")


h5f = h5py.File(output, 'w')
h5f.create_dataset('dataset_1', data = feats)
h5f.create_dataset('dataset_2', data = np.string_(names))
h5f.close()

Extracting features from image -  4.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 966ms/step
Extracting features from image -  horse2.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 589ms/step
Extracting features from image -  zebra2.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step
Extracting features from image -  chihuahua.JPG
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 505ms/step
Extracting features from image -  irish_terrier.JPG
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/step
Extracting features from image -  1.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 498ms/step
Extracting features from image -  horse1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 480ms/step
Extracting features from image -  2.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step
Extracting features from image -  tiger1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 501ms/step
Extracting features from image -  monkey1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 480ms/step
Extracting features from image -  monkey2.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 501ms/step
Extracting features from image -  zebra1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/

In [9]:
import h5py

import matplotlib.pyplot as plt

In [10]:
h5f = h5py.File("VGG16Features.h5",'r')
feats = h5f['dataset_1'][:]
imgNames = h5f['dataset_2'][:]
h5f.close()

In [11]:
queryImg = "query_images/monkey4.jpg"

In [14]:
X = model.extract_feat(queryImg)
len(X)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step


512

In [15]:
scores = []
from scipy import spatial
for i in range(feats.shape[0]):
    score = 1-spatial.distance.cosine(X, feats[i])
    scores.append(score)
scores = np.array(scores)
rank_ID = np.argsort(scores)[::-1]
rank_score = scores[rank_ID]

In [16]:
maxres = 3
imlist = [imgNames[index] for i,index in enumerate(rank_ID[0:maxres])]
print("top %d images in order are: " %maxres, imlist)

top 3 images in order are:  [b'monkey2.jpg', b'monkey1.jpg', b'chihuahua.JPG']
